<a href="https://colab.research.google.com/github/rahulpirwani7/Student-Pass-Fail-Prediction/blob/main/Passandfailpredication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("kundanbedmutha/exam-score-prediction-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'exam-score-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/exam-score-prediction-dataset


In [2]:
files = os.listdir(path)

print("\nFiles in the dataset:")
for file in files:
    print(file)

csv_file = None
for file in files:
    if file.endswith(".csv"):
        csv_file = file
        break

if csv_file is None:
    print("No CSV file found in the dataset.")
else:
    file_path = os.path.join(path, csv_file)
    df = pd.read_csv(file_path)


Files in the dataset:
Exam_Score_Prediction.csv


In [3]:
df.columns=df.columns.tolist()
df.drop('student_id',axis=1,inplace=True)


In [4]:
df

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7
...,...,...,...,...,...,...,...,...,...,...,...,...
19995,18,other,bba,6.50,71.3,yes,5.0,good,self-study,low,easy,86.5
19996,18,male,b.com,3.71,41.6,no,5.9,average,coaching,medium,moderate,60.9
19997,19,other,diploma,7.88,68.2,yes,4.6,poor,group study,low,easy,64.5
19998,19,male,bba,4.60,76.3,no,6.1,good,self-study,medium,moderate,79.0


In [5]:
df.describe()

,age,study_hours,class_attendance,sleep_hours,exam_score
count,20000.000000,20000.000000,20000.000000,20000.00000,20000.000000
mean,20.473300,4.007604,70.017365,7.00856,62.513225
std,2.284458,2.308313,17.282262,1.73209,18.908491
min,17.000000,0.080000,40.600000,4.10000,19.599000
25%,18.000000,2.000000,55.100000,5.50000,48.800000
50%,20.000000,4.040000,69.900000,7.00000,62.600000
75%,22.000000,6.000000,85.000000,8.50000,76.300000
max,24.000000,7.910000,99.400000,9.90000,100.000000


In [6]:
X=['gender','study_method','course']
df = pd.get_dummies(
    df,
    columns=X,
    prefix="",
    prefix_sep="",
    dtype=int
)

In [7]:
df

,age,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,facility_rating,exam_difficulty,exam_score,female,...,mixed,online videos,self-study,b.com,b.sc,b.tech,ba,bba,bca,diploma
0,17,2.78,92.9,yes,7.4,poor,low,hard,58.9,0,...,0,0,0,0,0,0,0,0,0,1
1,23,3.37,64.8,yes,4.6,average,medium,moderate,54.8,0,...,0,1,0,0,0,0,0,0,1,0
2,22,7.88,76.8,yes,8.5,poor,high,moderate,90.3,0,...,0,0,0,0,1,0,0,0,0,0
3,20,0.67,48.4,yes,5.8,average,low,moderate,29.7,0,...,0,1,0,0,0,0,0,0,0,1
4,20,0.89,71.6,yes,9.8,poor,low,moderate,43.7,1,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,18,6.50,71.3,yes,5.0,good,low,easy,86.5,0,...,0,0,1,0,0,0,0,1,0,0
19996,18,3.71,41.6,no,5.9,average,medium,moderate,60.9,0,...,0,0,0,1,0,0,0,0,0,0
19997,19,7.88,68.2,yes,4.6,poor,low,easy,64.5,0,...,0,0,0,0,0,0,0,0,0,1
19998,19,4.60,76.3,no,6.1,good,medium,moderate,79.0,0,...,0,0,1,0,0,0,0,1,0,0


In [8]:
df["Passed"] = df["exam_score"].apply(lambda x: "Passed" if x >= 65 else "Failed")
df.drop("exam_score", axis=1, inplace=True)

In [9]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
X=['internet_access','Passed']
for i in X:
    df[i]=le.fit_transform(df[i])

df["sleep_quality"] = df["sleep_quality"].map({
    "poor": 0,
    "average": 1,
    "good": 2
})

df['facility_rating']=df['facility_rating'].map({
    'low':0,
    'medium':1,
    'high':2
})

df['exam_difficulty']=df['exam_difficulty'].map({
    'easy':0,
    'moderate':1,
    'hard':2
})


In [10]:
from sklearn.model_selection import train_test_split
X = df.drop('Passed', axis=1)
y = df['Passed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

etas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]
iterations = [10, 50, 100, 500, 1000, 5000, 10000]

for eta in etas:
    print(f"\n========== eta0 = {eta} ==========")

    for it in iterations:
        model = SGDClassifier(
            loss="log_loss",
            learning_rate="constant",
            eta0=eta*1e-5,
            max_iter=it,
            tol=None,
            random_state=42
        )

        model.fit(X_train, y_train)

        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc = accuracy_score(y_test, model.predict(X_test))

        print(f"max_iter={it:5d} | Train={train_acc:.4f} | Test={test_acc:.4f}")


========== eta0 = 0.1 ==========
max_iter=   10 | Train=0.8349 | Test=0.8420
max_iter=   50 | Train=0.8354 | Test=0.8418
max_iter=  100 | Train=0.8349 | Test=0.8415
max_iter=  500 | Train=0.8364 | Test=0.8437
max_iter= 1000 | Train=0.8369 | Test=0.8433
max_iter= 5000 | Train=0.8380 | Test=0.8443
max_iter=10000 | Train=0.8385 | Test=0.8442

========== eta0 = 0.01 ==========
max_iter=   10 | Train=0.8346 | Test=0.8420
max_iter=   50 | Train=0.8347 | Test=0.8420
max_iter=  100 | Train=0.8349 | Test=0.8420
max_iter=  500 | Train=0.8354 | Test=0.8418
max_iter= 1000 | Train=0.8349 | Test=0.8415
max_iter= 5000 | Train=0.8364 | Test=0.8437
max_iter=10000 | Train=0.8369 | Test=0.8433

========== eta0 = 0.001 ==========
max_iter=   10 | Train=0.8346 | Test=0.8420
max_iter=   50 | Train=0.8346 | Test=0.8420
max_iter=  100 | Train=0.8346 | Test=0.8420
max_iter=  500 | Train=0.8347 | Test=0.8420
max_iter= 1000 | Train=0.8349 | Test=0.8420
max_iter= 5000 | Train=0.8354 | Test=0.8418
max_iter=10000 

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_prob = model.predict_proba(X_train)[:, 1]

threshold=0.5
y_pred = (y_prob >= threshold).astype(int)

print(40*" "+f"Threshold = {threshold}")
print(40*" "+"Accuracy :", accuracy_score(y_train, y_pred))
print(40*" "+"Precision:", precision_score(y_train, y_pred))
print(40*" "+"Recall   :", recall_score(y_train, y_pred))
print(40*" "+"F1-score :", f1_score(y_train, y_pred))
print("-" * 100)

                                        Threshold = 0.5
                                        Accuracy : 0.8346428571428571
                                        Precision: 0.8156028368794326
                                        Recall   : 0.8254017787486347
                                        F1-score : 0.8204730515703761
----------------------------------------------------------------------------------------------------


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_prob = model.predict_proba(X_test)[:, 1]

threshold=0.5
y_pred = (y_prob >= threshold).astype(int)

print(40*" "+f"Threshold = {threshold}")
print(40*" "+"Accuracy :", accuracy_score(y_test, y_pred))
print(40*" "+"Precision:", precision_score(y_test, y_pred))
print(40*" "+"Recall   :", recall_score(y_test, y_pred))
print(40*" "+"F1-score :", f1_score(y_test, y_pred))
print("-" * 100)

                                        Threshold = 0.5
                                        Accuracy : 0.842
                                        Precision: 0.8275101140125046
                                        Recall   : 0.8244778307072188
                                        F1-score : 0.8259911894273128
----------------------------------------------------------------------------------------------------


In [15]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

[[2802  469]
 [ 479 2250]]


In [16]:
df["Passed"].value_counts()

,count
Passed,
0,10862
1,9138


In [17]:
print(y_train.value_counts())

Passed
0    7591
1    6409
Name: count, dtype: int64


In [18]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(14000, 23)
(6000, 23)
(14000,)
(6000,)


In [19]:
print(X_train.head())

AttributeError: 'numpy.ndarray' object has no attribute 'head'

In [ ]:
print(df.corr(numeric_only=True)["Passed"].sort_values(ascending=False))